# Entrainement des modèles 

***Résumé Exécutif du Feature Engineering - Prévision Solaire J+1***

L'étape de feature_engineering a consisté à la prise en compte des conclusions de l'analyse exploratoire (EDA), ainsi que la création de garde-fous données manquantes/outliers ainsi que la création de features cycliques et de variables laggées.

**Données** : Dataset horaire sur 2.5 ans (01/2023 - 05/2025) de production RTE et prévisions OpenMétéo. Identification de 2 valeurs manquantes nocturnes dans la cible, qui seront imputées à 0.

**Gardes-fous :**
- Création de tests IQR/Z-score et raise des outliers en intersection des deux filtres ;
- Interpolation des séquences temporelles inférieures à 3h consécutives ;
- Lors d'une absence de séquence de plus de 3 heures, création d'un reporting des séquences les plus longues, et potentiellement création d'un futur algorithme KNN - Filtre de Kalman.

**Features créées :**
- Création de features cycliques heures + mois, en fonction de la saisonnalité du cycle solaire ;
- Création de features laggées (data leakage évité): 
  - Retard de 1, 6, 12, 24 et 48 (observation des cross-correlation + cohérent physiquement) ;
  - Moyennes mobiles de 6, 12, 24 et 48 périodes (analogue aux features retard) ;
  - Ramp de 6, 12, 24 et 48 périodes.

**Sélection des features :**
- Sélection de features agnostique pour LGBM et LSTM (Embedding via LightGBM) ;
 
**Etapes effectuées dans ce notebook** :

- Baseline SARIMAX avec les features physiques les plus corrélées ;
- Entrainement LGBM Validation croisée "Expanding Window" avec optimisation des hyperparamètres (Optuna) ;
- Entrainement LSTM et validation OOT ;
- Logging des modèles LSTM/LGBM sur le serveur MLflow
- Rédactions des data contracts et encapsulation des logiques production training

**Future works** :
- Walk-forward validation modèles LightGBM et LSTM (RMSE, MAE, Skill score vs persistance par horizon), avec boostraping sur le dataset test pour quantifier l'incertitude.

### Import des librairies

In [2]:
# Registry
import mlflow
from mlflow import lightgbm
from mlflow.pyfunc import PythonModel
from dotenv import dotenv_values

# ML / DL
import torch.nn as nn
import torch
from statsmodels.tsa.statespace.sarimax import SARIMAX
import numpy as np
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import optuna
from optuna.pruners import MedianPruner
import lightgbm as lgb
from torch.utils.data import DataLoader, TensorDataset

#Logs, typing, libraries standards
import traceback
import logging 
import os
from math import inf
import joblib
import warnings
import typing
from typing import Dict, List, Optional, Any, Tuple
from dataclasses import dataclass
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

# Config basique logging
PROJECT_ROOT = Path().resolve().parent
logging.basicConfig(filename=PROJECT_ROOT / "logs/data_training.log", level=logging.INFO, datefmt="%Y-%m-%d-%H-%M-%s")
logging.getLogger('mlflow').setLevel(logging.ERROR)

#### Préparation des datasets d'entrainement

La stratégie est la suivante : 
- SARIMAX dispose de la série brute en MWh en entrée, car ce n'est qu'une baseline ;
- La série brute est ensuite normalisée par le décile 99 glissant sur les 90 derniers jours. Il suit l'augmentation de la capacité du parc, et n'est que data-driven, nous assurant que la tendance est bien incluse dedans.

In [3]:
# Data core
df = pd.read_csv(PROJECT_ROOT / "data/processed/training_dataset.csv", index_col=0)
df.index = pd.to_datetime(df.index, utc=True).tz_convert("Europe/Paris")
meteo_features = joblib.load(PROJECT_ROOT / "data/processed/features/meteo_features.plk")

# Features
X = df.drop(columns="solar_mw")
y = df["solar_mw"]

# Séparation train test split
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.15,
                                                    shuffle=False, 
                                                    random_state=42)

# Configuratio secrets
config = dotenv_values(PROJECT_ROOT / ".env")

In [4]:
def init_mlflow(config: dict, experiment_name: str) -> None:
    """
    MLFlow initializing
    """
    os.environ["MLFLOW_TRACKING_USERNAME"] = config["MLFLOW_TRACKING_USERNAME"]
    os.environ["MLFLOW_TRACKING_PASSWORD"] = config["MLFLOW_TRACKING_PASSWORD"]
    mlflow.set_tracking_uri(config["MLFLOW_TRACKING_URI"])
    mlflow.set_experiment(experiment_name=experiment_name)

In [5]:
def rmse(y_true: pd.Series, y_pred: pd.Series) -> Any:
    """Return rmse of y_true and y_prec"""
    return np.sqrt(mean_squared_error(y_true, y_pred))

## Baseline SARIMAX

Le modèle SARIMAX est un modèle statistique qui servira de baseline de performance aux autres modèles.

In [6]:
def train_sarimax(X: pd.DataFrame, y: pd.Series,  nb_cv_splits: int, num_trials: Optional[int|None]) -> tuple:
    
    """Entrainement d'un modèle SARIMAX avec nombre d'essais pour optimisation.
    Retourne la quantification de son erreur (best_rmse) et ses hyperparamètres 
    optimaux (dict_best_params) sur un dataset de validation (X_val, y_val)

    Args:
        X_train (pd.DataFrame): Dataset d'entrainement
        y_train (pd.Series): Variable cible d'entrainement
        X_val (pd.DataFrame): Dataset de validation
        y_val (pd.Series): Variable cible de validation

    Returns:
        best_rmse (float), dict_best_params (Dict) : Erreur (best_rmse) 
        et ses hyperparamètres optimaux (dict_best_params)
    """
    # Initialisation
    folds = TimeSeriesSplit(n_splits=nb_cv_splits)

    def objective_sarimax(trial) -> np.float64 :
        """Prend en entrée un set d'hyperparamètres SARIMAX issus du sampler d'Optuna, 
        et retourne le RMSE associé.

        Args:
            trial : Set d'hyperparamètres SARIMAX

        Returns:
            rmse (np.float64): Listes des RMSE de la CV du modèle
        """
        rmses = []

        # hyperparamètres
        order = (trial.suggest_int("p", 0, 1),
            trial.suggest_int("d", 0, 1),
            trial.suggest_int("q", 0, 1))

        seasonal_order = (
            trial.suggest_int("P", 0, 1), 
            trial.suggest_int("D", 0, 1),
            trial.suggest_int("Q", 0, 1),
            24, #Saisonnalité fixe
        )
        
        for train_idx, val_idx in folds.split(X):
            
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            # Training
            try:
                logging.info(f"Lancement de l'entrainement sur les paramètres : {order}\n{seasonal_order}")
                model = SARIMAX(endog=y_train, 
                                exog=X_train, 
                                order = order, 
                                seasonal_order = seasonal_order,
                                enforce_stationarity=False,
                                enforce_invertibility=False)
                fitted_model = model.fit(disp=False)
                logging.info(f"Prédiction du modèle : {order}\n{seasonal_order}")
                
                # Prédictions sur validation
                prediction_interval = fitted_model.get_prediction(start=y_val.index[0],
                                                          end=y_val.index[-1],
                                                          exog=X_val)
                predictions = prediction_interval.predicted_mean
                rmse = np.sqrt(mean_squared_error(y_val, predictions))
                rmses.append(rmse)
            
            except Exception as e:
                logging.info("Echec de l'entrainement du modèle :", e)
                logging.debug("Détails complets :\n%s", traceback.format_exc())
                return inf

        return np.mean(rmses)
    
    # Recherche des HP et prédictions
    try:
        study = optuna.create_study(direction="minimize")
        study.optimize(objective_sarimax, n_trials=num_trials, n_jobs=1)
    
    except Exception as e:
        logging.error(
        "Échec de la recherche d’HP. Paramètres : n_trials=%d, n_jobs=%d. Erreur : %s",
        num_trials, 4, str(e),
        exc_info=True)
        raise
    
    return study.best_value, study.best_params

In [9]:
# A lancer
features_sarimax = ["global_tilted_irradiance", 
                    "hour_cos", 
                    "hour_sin"]

X_sarimax_train = X_train[features_sarimax]


num_trials = 10
best_value_rmse, best_params_sarimax = train_sarimax(X=X_sarimax_train, 
                                                        y=y_train,
                                                        nb_cv_splits=3,
                                                        num_trials=num_trials, 
                                                        )

[I 2026-04-13 14:01:56,717] A new study created in memory with name: no-name-c11ced10-1a55-48b7-9e97-c24792270d20
[I 2026-04-13 14:12:40,815] Trial 0 finished with value: 1786.3996494178537 and parameters: {'p': 0, 'd': 1, 'q': 1, 'P': 0, 'D': 1, 'Q': 1}. Best is trial 0 with value: 1786.3996494178537.
[I 2026-04-13 14:28:50,327] Trial 1 finished with value: 1391.771052002918 and parameters: {'p': 1, 'd': 1, 'q': 1, 'P': 1, 'D': 1, 'Q': 1}. Best is trial 1 with value: 1391.771052002918.
[I 2026-04-13 14:41:50,696] Trial 2 finished with value: 297.73797615268865 and parameters: {'p': 0, 'd': 0, 'q': 0, 'P': 1, 'D': 1, 'Q': 1}. Best is trial 2 with value: 297.73797615268865.
[I 2026-04-13 14:43:45,196] Trial 3 finished with value: 822.9021811296811 and parameters: {'p': 1, 'd': 1, 'q': 0, 'P': 1, 'D': 0, 'Q': 0}. Best is trial 2 with value: 297.73797615268865.
[I 2026-04-13 14:45:26,768] Trial 4 finished with value: 608.3509374665431 and parameters: {'p': 1, 'd': 0, 'q': 1, 'P': 0, 'D': 

## Production models

### Preprocessing standardization LGBM LSTM

#### Data driven method : preprocessing

In [6]:
class  SolarDataProcessor(BaseEstimator, TransformerMixin):
    
    """Data preprocessing before model inference :
    - Target features normalization by the 99th quantile to detrend the time serie (last quantile for validation/prod and series for training);
    - Scaling (MinMax) the others features to increase the LSTM convergence (if used)
    """

    def __init__(
            self,  
            meteo_features: List[str], 
            window_days: float = 90, 
            quantile: float = 0.99,
            annot: str = "mw",
            selected_features: Optional[List[str]] = None
            ):
        
        self.meteo_features = meteo_features
        self.window_days = window_days
        self.quantile = quantile
        self.meteo_scaler = MinMaxScaler()
        self.annot = annot
        self.selected_features = selected_features

    def _rolling_quantile(
            self, 
            y: pd.Series) -> pd.Series:
        
        """Return the 99th rolling quantile on window days"""
        
        window_hours = int(self.window_days * 24)

        return y.rolling(window=window_hours, min_periods=24).quantile(self.quantile).bfill().astype(np.float32)
        

    def fit(self, X: pd.DataFrame, y: pd.Series):
        
        # Meteo features fit
        self.effective_meteo_features_ = [f for f in self.meteo_features if f in X.columns]
        if self.effective_meteo_features_:
            self.meteo_scaler.fit(X[self.effective_meteo_features_])

        if y is not None:   
            # Quantile fit
            full_denominators = self._rolling_quantile(y)
            self.denominator_series_ = full_denominators[~full_denominators.index.duplicated(keep='last')] # Eviter les doubles index
            self.last_denominator_ = float(full_denominators.iloc[-1])

        return self
    
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:

        X_res = X.copy()

        # MinMax meteo features scaling
        to_scale = [f for f in self.effective_meteo_features_ if f in X.columns]
        if to_scale:
            X_res[to_scale] = self.meteo_scaler.transform(X[to_scale])

        # rolling quantile if not training, last quantile if not
        denoms = (
            self.denominator_series_
            .reindex(X_res.index)
            .fillna(self.last_denominator_)
            .values
        )
            
        # Derivated target features scaling
        mw_columns = [col for col in X_res.columns if self.annot in col]
        for col in mw_columns:
            X_res[col] = X_res[col] / denoms
        
        
        if self.selected_features is not None:
            return X_res[[c for c in self.selected_features if c in X_res.columns]]   
        
        return X_res
    
    def transform_y(self, y: pd.Series) -> pd.Series:
        
        denoms = (
            self.denominator_series_
            .reindex(y.index)
            .fillna(self.last_denominator_)
            .values)
        
        return y / denoms
        
    def inverse_transform_y(self, y_pred: np.ndarray, input_index: pd.Index) -> np.ndarray:
        """Return y in MWh. Must the index of infered or trained input to reindex quantiles"""
        
        denoms = (
            self.denominator_series_
            .reindex(input_index)
            .fillna(self.last_denominator_)
            .values
        )
        return y_pred * denoms

## Modèle LightGBM

Le RMSE sera pris comme métrique de minimisation pour permettre une meilleure prise en compte des évènements extrêmes. Le MAE, MAPE et SMAPE seront aussi retournés, cette fois à titre indicatif pour comparaison.

In [7]:
# Data contract
@dataclass
class HorizonRunResult:
    """Data contract between ML and MLflow logs"""
    # Identifiers
    horizon:  int
    set_name: str
    
    # ML results
    model:        Pipeline
    best_rmse:    float
    best_params:  dict
    X_train:      pd.DataFrame

    # --- Properties ---

    @property
    def n_features(self) -> int:
        return self.X_train.shape[1]
    
    @property
    def params_to_log(self) -> Dict[str, Optional[float|str]]:
        """HP and experimental context (MLflow log_params)"""

        return {
            **self.best_params,
            "horizon":     self.horizon,
            "feature_set": self.set_name,
            "n_features" : self.n_features,
        }

    @property
    def tags_to_log(self) -> Dict[str, str]:
        """
        Tags to log to MLFlow server
        """

        return {
            "project"       : "solar_forecast",
            "model_family"  : "lightgbm",
            "training_type" : "multihorizon",
            "horizon_bucket":  self._horizon_bucket(),
            "feature_set"   :  self.set_name,
            "is_baseline"   :  str(self.set_name == "short"),
        }
    
    @property
    def signature(self):
        return mlflow.models.infer_signature(
            self.X_train,
            self.model.predict(self.X_train)
        )
    
    def to_record(self) -> Dict[str, Optional[float|str]]:
        """Return results dict"""
        return {
            "horizon":     self.horizon,
            "feature_set": self.set_name,
            "n_features":  self.n_features,
            "rmse":        self.best_rmse,
            **self.best_params,
        }
    
    def _horizon_bucket(self) -> str:
        """Easy filter"""
        if self.horizon <= 6:   
            return "short"
        if self.horizon <= 12:  
            return "medium"
        return "long"

### Entrainement du modèle

In [8]:
def log_horizon_run(result: HorizonRunResult) -> None:
    """MLflow logs and artifact"""
    
    # Tags - params - metrics - artifact
    mlflow.set_tags(result.tags_to_log)
    mlflow.log_params(result.params_to_log)
    mlflow.log_metric("best_rmse_val", result.best_rmse)
    lightgbm.log_model(
        result.model,
        name="model",
        signature=result.signature,
        input_example=result.X_train.iloc[:3]
    )

In [9]:
def prepare_horizon_data(X: pd.DataFrame, y: pd.Series, horizon: int) -> Tuple[pd.DataFrame, pd.Series]:
    """Return shift + alignment for a given horizon"""
    
    # Init
    y_shifted = y.shift(-horizon).dropna()
    X_aligned = X.loc[y_shifted.index]

    return X_aligned, y_shifted

In [10]:
def train_lightgbm(X: pd.DataFrame, 
                   y: pd.Series, 
                   meteo_features: List[str],
                   nb_cv_splits: int,
                   num_trials: Optional[int|None]
                   ) -> Tuple[float, Dict[Any, Any]]:
    
    """Entrainement d'un modèle LightGBM avec nombre d'essais pour optimisation.
    Retourne la quantification de son erreur (best_rmse) et ses hyperparamètres 
    optimaux (dict_best_params) sur un dataset de validation (X_val, y_val)

    Args:
        X_train (pd.DataFrame): Dataset d'entrainement
        y_train (pd.Series): Variable cible d'entrainement
        X_val (pd.DataFrame): Dataset de validation
        y_val (pd.Series): Variable cible de validation

    Returns:
        best_rmse (float), dict_best_params (Dict) : Erreur (best_rmse) 
        et ses hyperparamètres optimaux (dict_best_params)
    """

    # Métriques scorer et folds
    rmse_scorer = make_scorer(rmse, greater_is_better=False)
    tscv = TimeSeriesSplit(gap=0, n_splits=nb_cv_splits)

    def objective_lightgbm(trial) -> np.float64 :
        """Prend en entrée un set d'hyperparamètres LightGBM issus du sampler d'Optuna, 
        et retourne le RMSE associé.

        Args:
            trial : Set d'hyperparamètres LightGBM

        Returns:
            rmse (np.float64): Racine carrée de l'erreur quadratique moyenne du modèle
        """
        
        # HP
        params = {
            "num_leaves" : trial.suggest_int("num_leaves", 10, 100),
            "learning_rate" : trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            "max_depth" : trial.suggest_int("max_depth", 1, 30),
            "n_estimators" : trial.suggest_int("n_estimators", 100, 1000),
            "min_child_samples" : trial.suggest_int("min_child_samples", 10, 50),
            "verbosity" : -1,
            "random_state": 42
        }

        
        # Cross validation manuelle car cross_val_score ne gère pas inverse_transform de y
        scores = []
        for train_idx, val_idx in tscv.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            # Model fit avec les bonnes valeurs
            # y normalization outer : La pipeline ne transforme pas la target
            final_processor = SolarDataProcessor(meteo_features=meteo_features)
            final_processor.fit(X_train, y_train)
            X_train_scaled = final_processor.transform(X=X_train)
            y_train_scaled = final_processor.transform_y(y=y_train)

            # Model fit
            final_model = lgb.LGBMRegressor(**params)
            final_model.fit(X_train_scaled, y_train_scaled)
            
            # Pipeline already fitted
            pipeline = Pipeline([
            ('processor', final_processor),
            ('model', final_model)
        ])
        
            # Fitting total
            norm_y_pred = pipeline.predict(X_val)

            # On compare des MWh avec des MWh
            y_pred = pipeline.named_steps['processor'].inverse_transform_y(norm_y_pred, y_val.index) 
            
            scores.append(rmse(y_true=y_val, y_pred=y_pred))

        return np.mean(scores)
    
    # Recherche des HP et prédiction
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="minimize",
                                pruner=MedianPruner(
                                    n_startup_trials=5, 
                                    n_warmup_steps=3)
                                )
    
    study.optimize(objective_lightgbm, n_trials=num_trials, n_jobs=-1)

    return study.best_value, study.best_params

In [11]:
def fit_best_model(X: pd.DataFrame, 
                   y: pd.Series, 
                   meteo_features: List[str],
                   selected_features: List[str],
                   n_cv_splits: int, 
                   num_trials: Optional[int|None]) -> Tuple[Pipeline, float, dict]:
    """
    Return (model, best_rmse, best_params).
    """
    
    # HP Searching
    best_rmse, best_params = train_lightgbm(
        X=X, y=y,
        meteo_features=meteo_features,
        nb_cv_splits=n_cv_splits,
        num_trials=num_trials,
    )

    # Model fit avec les bonnes valeurs
    final_processor = SolarDataProcessor(
        meteo_features=meteo_features, 
        selected_features=selected_features)
    
    final_processor.fit(X, y)
    X_train_scaled = final_processor.transform(X=X)
    y_train_scaled = final_processor.transform_y(y=y)
    
    # Fit model
    final_model = lgb.LGBMRegressor(**best_params, verbosity=-1, random_state=42)
    final_model.fit(X_train_scaled, y_train_scaled)

    # Pipeline already fitted
    pipeline = Pipeline([
    ('processor', final_processor),
    ('model', final_model)
    ])

    return pipeline, best_rmse, best_params

In [12]:
def training_multioutput_lgbm_model(X: pd.DataFrame, 
                                    y: pd.Series, 
                                    meteo_features: List[str],
                                    n_horizons: int, 
                                    n_cv_splits: int, 
                                    num_trials: Optional[int|None],
                                    feature_sets: Dict[str, List[str]]) -> Dict[str, Pipeline]:
    
    """Train n horizons LGBM and log params, metrics and tags on cloud MLFlow server.

    Args:
        X (pd.DataFrame): Train dataset
        y (pd.Series): Target feature
        n_horizons (int): Horizon number to train
        n_cv_splits (int): Split nomber for time series cross validation
        num_trials (Optional[int | None]): Trial Optuna number
    """

    models_dict = {}
    for horizon in range(1, n_horizons+1):
        logging.info(f'--- Training horizon t+{horizon} ---')
        
        X_h, y_h = prepare_horizon_data(
            X=X, 
            y=y, 
            horizon=horizon
        )
        
        # Features horizon group
        if horizon <= 3:
            set_name = "short"
        elif horizon <= 12:
            set_name = "mid"
        else:
            set_name = "long"

        # Training 
        features = feature_sets[set_name]
        model, best_rmse, best_params = fit_best_model(
            X=X_h[features], y=y_h,
            meteo_features=meteo_features,
            selected_features=features,
            n_cv_splits=n_cv_splits,
            num_trials=num_trials
        )

        # Save for wrap
        models_dict[f'h+{horizon}'] = model

        # MLFlow
        logging.info(f'Best model fitted on {set_name}- Data contract loading')
        result = HorizonRunResult(
            horizon=horizon,
            set_name=set_name,
            best_rmse=best_rmse,
            best_params=best_params,
            model=model,
            X_train=X_h[features]
        )

        with mlflow.start_run(run_name=f"h+{horizon}_{set_name}", nested=True):
            log_horizon_run(result=result)
    
    return models_dict

In [13]:
# Parameters
n_cv_splits = 5
num_trials = 30
n_horizons = 24
selected_features = joblib.load(PROJECT_ROOT / "data/processed/features/lgbm_selected_features.plk")
init_mlflow(config=config, experiment_name="LGBM_Direct_Output_v1.1-stub")

In [14]:
# Training
models_dict = training_multioutput_lgbm_model(
        X=X_train, 
        y=y_train, 
        meteo_features=meteo_features,
        n_horizons=n_horizons,
        n_cv_splits=n_cv_splits, 
        num_trials=num_trials,
        feature_sets=selected_features
        )

🏃 View run h+1_short at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8/runs/f1807a378fd44dc1988c0a9572fdf64f
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8


🏃 View run h+2_short at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8/runs/b6be9dad21534ddd9dec07e0b08b9740
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8


🏃 View run h+3_short at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8/runs/028698dde9b14a3bb63fe0312065dcc5
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8


🏃 View run h+4_mid at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8/runs/fb23a094577e4457879f24f55b762e4e
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8


🏃 View run h+5_mid at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8/runs/e5d6500e849642088be460e7ffd7bf65
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8


...

🏃 View run h+20_long at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8/runs/1a855f17830942419915410322f5b2d3
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8


🏃 View run h+21_long at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8/runs/37caf044d05c44d38464ae1e9c93e8fa
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8


🏃 View run h+22_long at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8/runs/d12f59be51614aa2afd92478aaebfe49
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8


🏃 View run h+23_long at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8/runs/f7d80ce99c7044bdb2e2f2bf91d5693b
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8


🏃 View run h+24_long at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8/runs/aef169df940e4a2a88c9b3b9cef6c925
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8


### Wrapper LGBM Multioutput

Les 24 modèles sont stockés sur MLFlow, mais pour la production il est nécessaire de produire l'artifact des 24 modèles d'un coup.

In [15]:
class MultiHorizonLGBMWrapper(PythonModel):
    
    def __init__(self, models_dict: Dict[str, Pipeline], num_horizons: int):
        self.models_dict = models_dict
        self.num_horizons = num_horizons
        
    def predict(self, model_input: pd.DataFrame): # type: ignore

        all_predictions = {}
        future_index = model_input.index

        for h in range(1, self.num_horizons+1):
            key = f"h+{h}"
            if key in self.models_dict:
                pipe = self.models_dict[key]
                norm_pred = pipe.predict(model_input)
                all_predictions[key] = pipe.named_steps['processor'].inverse_transform_y(norm_pred, future_index)
    
        return pd.DataFrame(all_predictions, index=model_input.index).clip(lower=0)

In [16]:
init_mlflow(config=config, experiment_name="LGBM_Direct_Output_v1.1-stub")
meta_model = MultiHorizonLGBMWrapper(models_dict=models_dict, num_horizons=24)

with mlflow.start_run(run_name="LGBM_DirectPrediction_J+1"):
    input_example = X_train.iloc[:3] 
    
    # Log du modèle personnalisé
    mlflow.pyfunc.log_model(
        name="wrapped_denormalized_lgbm_model",
        python_model=meta_model,
        input_example=input_example
    )

🏃 View run LGBM_DirectPrediction_J+1 at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8/runs/8b9e1aca07f74e6989e3310ce323cd08
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/8


## Modèle LSTM seq2seq

Il est proposé comme benchmark DL d'utiliser un seq2seq afin de prendre en compte de manière intéressante la structure historique/prévision de la production solaire.

### Dataloader

On propose un dataloader avec un stride de 1 pour créer le plus de séquence possible. la séquence est égale à 48, soit 2 jours d'historiques pour prédire 1 jour ensuite.

In [29]:
def to_dataloader(
        training_dataset: pd.DataFrame, 
        encoder_features: List[str],
        decoder_features: List[str],
        target: pd.Series,
        seq_length: int,
        seq_future_length: int, 
        batch_size: int,
        stride: int
        ):
    
    """Construit un DataLoader Seq2Seq avec séparation encoder/decoder.
    
    Args:
        training_dataset: DataFrame complet des features
        encoder_features: Features passées (autorégressives + cycliques) 
        decoder_features: Features futures connues (météo + cycliques)
        target: Série cible (solar_mw normalisé)
        seq_length: Longueur séquence encodeur
        seq_future_length: Longueur séquence décodeur (horizons)
        batch_size: Taille des batchs
        stride: Pas entre les fenêtres
    
    Returns:
        DataLoader PyTorch

    """

    #Init
    L_total = training_dataset.shape[0]
    total_window_length = seq_length + seq_future_length
    indices = np.arange(0, L_total-total_window_length+1, stride, dtype=int)

    # Découpage des indices
    indices_past = indices[:, None] + np.arange(seq_length) 
    indices_future = indices[:, None] + np.arange(seq_length, (seq_length+seq_future_length)) 
    
    # encoder: (features autorégressives + cycliques + y)
    partial_encoder = training_dataset[encoder_features]
    encoder_dataset = pd.concat([partial_encoder, target], axis=1)
    X_past = torch.tensor(
        encoder_dataset.values[indices_past], dtype=torch.float32
    )

    # eecoder: futur (prévisions + encodages cycliques)
    X_future = torch.tensor(
        training_dataset[decoder_features].values[indices_future], dtype=torch.float32
    )

    # target
    y_target = torch.tensor(
        target.values[indices_future], dtype=torch.float32
    ).unsqueeze(-1)

    # Tensor final
    final_dataset = TensorDataset(X_past, X_future, y_target)
    dataloader = DataLoader(final_dataset, batch_size=batch_size, shuffle=False)
    
    # Asserts
    assert X_past.shape[2]   == len(encoder_features) + 1, \
        f"Encoder shape mismatch: {X_past.shape[2]} - {len(encoder_features) + 1}"
    assert X_future.shape[2] == len(decoder_features), \
        f"Decoder shape mismatch: {X_future.shape[2]} - {len(decoder_features)}"
    
    return dataloader

### Seq2seq

In [30]:
class Encoder(nn.Module):
    
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, dropout: float):
        
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #Encoder
        self.encoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        
        #LayerNorm
        self.layer_norm = nn.LayerNorm(normalized_shape=hidden_size)

    def forward(self, x_past: np.ndarray):
        #x de la forme (batch_number, seq_length, all_features)
        _, (h_n, c_n) = self.encoder(x_past)
        h_n_norm = self.layer_norm(h_n) 
        c_n_norm = self.layer_norm(c_n)

        return (h_n_norm, c_n_norm)

In [31]:
class Decoder(nn.Module):

    def __init__(self, input_size: int, hidden_size: int, num_layers: int, output_len: int, dropout: float):
        
        super(Decoder, self).__init__()
        self.output_len = output_len # M=24

        #Decoder
        self.decoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        #Couche linéaire pour la sortie en prediction
        self.fc_out = nn.Linear(hidden_size, 1)


    def forward(self, x_future: np.ndarray, h_init_norm: np.ndarray, c_init_norm: np.ndarray):
        #x_future de la forme (batch_number, output_len, feature.difference(y))
        output, _ = self.decoder(x_future, (h_init_norm, c_init_norm))

        #Couche linéaire
        predictions = self.fc_out(output)

        return predictions

In [32]:
class Seq2seq(nn.Module):
    
    def __init__(
            self, 
            past_features_length: int, 
            future_features_length: int, 
            hidden_size: int, 
            num_layers: int, 
            output_len: int, 
            dropout: float = 0.0
        ):
        
        super(Seq2seq, self).__init__()
        
        self.encoder = Encoder(
            input_size=past_features_length, 
            hidden_size=hidden_size, 
            num_layers=num_layers,
            dropout=dropout
        )
        
        self.decoder = Decoder(
            input_size=future_features_length,
            hidden_size=hidden_size,
            num_layers=num_layers,
            output_len=output_len,
            dropout=dropout
        )

    def forward(self, x_past, x_future):
        #Encoder
        (h_n_norm, c_n_norm) = self.encoder(x_past)
        predictions = self.decoder(x_future, h_n_norm, c_n_norm)

        return predictions

### Entrainement du modèle

In [33]:
def train_seq2seq(model, 
                       train_dataloader: DataLoader, 
                       val_dataloader: DataLoader,
                       criterion,
                       optimizer, 
                       device,
                       num_epochs: int):
    """
    Trains a seq2seq model and evaluates it on a validation set each epoch.

    Args:
        model (torch.nn.Module): Seq2seq model.
        train_dataloader (DataLoader): Training data loader.
        val_dataloader (DataLoader): Validation data loader.
        criterion: Loss function.
        optimizer: Optimization algorithm.
        device (torch.device): CPU or GPU device.
        num_epochs (int): Number of training epochs.

    Returns:
        tuple: (train_losses, val_losses), lists of RMSE per epoch.
    """
    
    train_losses = []
    val_losses = []

    logging.info(f"Début de l'entrainement sur {num_epochs} sur {device}")
    
    for epoch in range(num_epochs):
        
        # --- Entrainement ---

        model.train()
        running_loss = 0.0

        for X_past, X_future, Y_target in train_dataloader:
            X_past, X_future, Y_target = X_past.to(device), X_future.to(device), Y_target.to(device)

            #Mise à zéro gradient
            optimizer.zero_grad()

            # Forward et perte
            Y_pred = model(X_past, X_future)
            loss = criterion(Y_pred, Y_target)

            # Rétropropagation et optimisation
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        
        # Statistiques
        avg_train_loss = running_loss / len(train_dataloader)
        train_losses.append(np.sqrt(avg_train_loss))
        
        # --- Evaluation sur validation dataset ---

        model.eval()
        validation_loss = 0.0

        with torch.no_grad():
            for X_past_val, X_future_val, Y_target_val in val_dataloader:
                X_past_val, X_future_val, Y_target_val = X_past_val.to(device), X_future_val.to(device), Y_target_val.to(device)
                
                Y_pred_val = model(X_past_val, X_future_val)
                loss_val = criterion(Y_pred_val, Y_target_val)

                validation_loss += loss_val.item()
                
        avg_val_loss = validation_loss / len(val_dataloader) # type: ignore
        val_losses.append(np.sqrt(avg_val_loss))

        print(f"Epoch [{epoch+1}/{num_epochs}] | Train RMSE: {np.sqrt(avg_train_loss):.6f} | Val RMSE: {np.sqrt(avg_val_loss):.6f}")
        
    return train_losses, val_losses

### Data preprocessing

In [34]:
def prepare_lstm_data(
    X: pd.DataFrame,
    y: pd.Series,
    meteo_features: List[str],
    encoder_features: List[str],
    decoder_features: List[str],
    seq_length: int,
    output_len: int,
    batch_size: int,
    stride: int,
    val_ratio: float = 0.25) -> Tuple[DataLoader, DataLoader, Any, pd.DataFrame, pd.Series]:
    
    """
    Prepares train/validation DataLoaders for a Seq2Seq LSTM model.

    Splits the dataset, fits a preprocessing pipeline on training data, scales features
    and target, and builds sliding-window sequences for encoder-decoder training.

    Args:
        X (pd.DataFrame): Input features.
        y (pd.Series): Target variable.
        meteo_features (List[str]): Meteorological feature names.
        encoder_features (List[str]): Features used by the encoder.
        decoder_features (List[str]): Features used by the decoder.
        seq_length (int): Length of input sequence.
        output_len (int): Length of prediction horizon.
        batch_size (int): Batch size for DataLoaders.
        stride (int): Step size for sliding windows.
        val_ratio (float, optional): Fraction of data used for validation. Defaults to 0.25.

    Returns:
        Tuple[DataLoader, DataLoader, Any, pd.DataFrame, pd.Series]:
            train_dl: Training DataLoader.
            val_dl: Validation DataLoader.
            processor: Fitted SolarDataProcessor.
            enc_example: Example encoder input (for model signature).
            dec_example: Example decoder target (for model signature).
    """
 
    # Init
    split_idx = int(len(X) * (1 - val_ratio))
    X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]
 
    # Preprocessing
    all_lstm_features = list(set(encoder_features) | set(decoder_features))
    processor = SolarDataProcessor(meteo_features=meteo_features, selected_features=all_lstm_features)  
    processor.fit(X_train, y_train)
    
    # Scaling
    X_train_scaled  = processor.transform(X_train)
    X_val_scaled = processor.transform(X_val)
    y_train_scaled  = processor.transform_y(y_train)
    y_val_scaled = processor.transform_y(y_val)
 
    # Dataloaders
    train_dl = to_dataloader(  
        training_dataset=X_train_scaled,
        encoder_features=encoder_features,
        decoder_features=decoder_features,
        target=y_train_scaled,
        seq_length=seq_length,
        seq_future_length=output_len,
        batch_size=batch_size,
        stride=stride,
    )
    val_dl = to_dataloader(
        training_dataset=X_val_scaled,
        encoder_features=encoder_features,
        decoder_features=decoder_features,
        target=y_val_scaled,
        seq_length=seq_length,
        seq_future_length=output_len,
        batch_size=batch_size,
        stride=stride,
    )
    
    # infer_sgignature
    example_end    = seq_length + output_len
    X_example      = X_train.iloc[:example_end]      # (seq+output, 346)
    y_past_example = y_train.iloc[:seq_length]        # MWh
 
    return train_dl, val_dl, processor, X_example, y_past_example

### Wrapping LSTMSeq2seq

In [35]:
@dataclass
class LSTMRunResult:
    """Data contract entre l'entraînement LSTM et MLflow"""
 
    # Params
    num_epochs:  int
    seq_length:  int    # longueur encodeur
    output_len:  int    # longueur décodeur = nb horizons
    model:        nn.Module
    best_rmse:    float
    best_params:  Dict[str, Any]
    encoder_features: List[str]
    decoder_features: List[str]
    processor: SolarDataProcessor
 
    # Signature
    X_example: pd.DataFrame   # shape (seq_length, n_encoder_features+1 (target))
    y_past_example: pd.DataFrame   # shape (output_len,  n_decoder_features)
 
    @property
    def params_to_log(self) -> Dict[str, Any]:
        return {
            **self.best_params,
            "num_epochs": self.num_epochs,
            "seq_length": self.seq_length,
            "output_len": self.output_len,
        }
 
    @property
    def tags_to_log(self) -> Dict[str, str]:
        return {
            "project":       "solar_forecast",
            "model_family":  "LSTMSeq2Seq",
            "framework":     "pytorch",
            "training_type": "seq2seq",
        }
 
    @property
    def signature(self) -> mlflow.models.ModelSignature:
        """MLflow signature (no inference)."""

        input_example = {
            "X": self.X_example.values.astype(np.float32),
            "y_past": self.y_past_example.values.astype(np.float32),
        }

        output_example = pd.DataFrame({
            "horizon": np.arange(1, self.output_len + 1, dtype=np.int32),
            "prediction_mw": np.zeros(self.output_len, dtype=np.float32),
        })

        return mlflow.models.infer_signature(
            model_input=input_example,
            model_output=output_example
        )

In [36]:
class SolarLSTMWrapper(PythonModel):
    """Wrapper PythonModel pour LSTM Seq2Seq"""
 
    def __init__(
            self, 
            processor, 
            model: nn.Module, 
            encoder_features: List[str],
            decoder_features: List[str],
            seq_length: int,
            output_len: int,
            device: str = "cpu"       
    ):
        
        self.processor        = processor
        self.model            = model
        self.encoder_features = encoder_features
        self.decoder_features = decoder_features
        self.seq_length       = seq_length
        self.output_len       = output_len
        self.device           = torch.device(device)
    
    def predict(self, model_input: list[typing.Dict[str, Optional[pd.DataFrame|pd.Series]]]) -> np.ndarray:
        
        """Prédit 24 horizons à partir d'une séquence encoder+decoder.
 
        Args:
        model_input: dict avec :
            - "X"      : DataFrame (seq_length + output_len) lignes*346 colonnes
            - "y_past" : Series ou array (seq_length,) — valeurs observées de solar_mw
 
        Returns:
        - preds (np.ndarray): predictions LSTM

        """
        # Init
        X      = model_input["X"]
        y_past = model_input["y_past"]
        assert len(X) == self.seq_length + self.output_len
        assert len(y_past) == self.seq_length

        # Scaling 
        X_scaled = self.processor.transform(X)

        # Slicing
        X_past_scaled   = X_scaled[self.encoder_features].iloc[:self.seq_length]
        X_future_scaled = X_scaled[self.decoder_features].iloc[self.seq_length:]

        # y_past ajouté à l'encodeur (normalisé)
        y_past_series = pd.Series(y_past, index=X.index[:self.seq_length])
        y_past_scaled = self.processor.transform_y(y_past_series)
        enc_array = np.concatenate(
            [X_past_scaled.values, y_past_scaled.values.reshape(-1, 1)], axis=1
        )

        # Inférence
        enc = torch.tensor(enc_array, dtype=torch.float32).unsqueeze(0).to(self.device)
        dec = torch.tensor(X_future_scaled.values, dtype=torch.float32).unsqueeze(0).to(self.device)
        self.model.eval()
        with torch.no_grad():
            norm_preds = self.model(enc, dec)   # (1, output_len, 1)

        # Return
        return np.clip(self.processor.inverse_transform_y(norm_preds.squeeze().cpu().numpy(), X_past_scaled.index), 0, None)

### Pipeline

Cette dernière partie propose un log du modèle LSTM sur MLFlow ainsi que le pipeline complet :
- initialisation,
- scaling,
- instanciation,
- training,
- envoi à MLflow.

In [37]:
def log_lstm_run(result: LSTMRunResult) -> None:
    """Logs tags, params, metrics and artifact LSTM on MLflow server"""
    
    mlflow.set_tags(result.tags_to_log)
    mlflow.log_params(result.params_to_log)
    mlflow.log_metric("best_rmse_val", result.best_rmse)
 
    wrapped_model = SolarLSTMWrapper(
        processor=result.processor,
        model=result.model,
        encoder_features=result.encoder_features,
        decoder_features=result.decoder_features,
        seq_length=result.seq_length,
        output_len=result.output_len,
        device="cpu",
    )
 
    mlflow.pyfunc.log_model(
        name="lstm_seq2seq_model",
        python_model=wrapped_model,
        signature=result.signature,
        input_example={
            "X":      result.X_example.values.astype(np.float32),
            "y_past": result.y_past_example.values.astype(np.float32),
        },
    )

In [38]:
def run_lstm_pipeline(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    meteo_features: List[str],
    encoder_features: List[str],
    decoder_features: List[str],
    lstm_params: Dict[str, Any],
) -> None:
    
    """Complete pipeline LSTM : DataLoaders → training → MLflow.
 
    Call train_seq2seq() then log on MLflow server.
    """
    seq_length = lstm_params["seq_length"]
    output_len = lstm_params["output_len"]
    batch_size = lstm_params["batch_size"]
    stride     = lstm_params["stride"]
    num_epochs = lstm_params["num_epochs"]
 
    # 1. Préparation données
    train_dl, val_dl, processor, X_ex, y_past_ex = prepare_lstm_data(
        X=X_train,
        y=y_train,
        meteo_features=meteo_features,
        encoder_features=encoder_features,
        decoder_features=decoder_features,
        seq_length=seq_length,
        output_len=output_len,
        batch_size=batch_size,
        stride=stride,
    )
 
    # 2. Instanciation modèle
    past_features_length   = len(encoder_features) + 1
    future_features_length = len(decoder_features)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
    model = Seq2seq(  # noqa: F821
        past_features_length=past_features_length,
        future_features_length=future_features_length,
        hidden_size=lstm_params["hidden_size"],
        num_layers=lstm_params["num_layers"],
        output_len=output_len,
        dropout=lstm_params["dropout"],
    ).to(device)
 
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
 
    # 3. Training (déjà dans le notebook)
    train_losses, val_losses = train_seq2seq(  # noqa: F821
        model=model,
        train_dataloader=train_dl,
        val_dataloader=val_dl,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=num_epochs,
    )
    best_rmse = min(val_losses)
 
    # 4. Data contract
    result = LSTMRunResult(
        num_epochs=num_epochs,
        seq_length=seq_length,
        output_len=output_len,
        model=model.to("cpu"),
        best_rmse=best_rmse,
        best_params=lstm_params,
        X_example=X_ex,
        y_past_example=y_past_ex,
        encoder_features=encoder_features,
        decoder_features=decoder_features,
        processor=processor,
    )
 
    # 5. MLflow
    with mlflow.start_run(run_name="LSTM_Seq2Seq_J+1"):
        log_lstm_run(result=result)

In [39]:
# Initialisation
encoder_cols = joblib.load(PROJECT_ROOT / "data/processed/features/encoder_features.plk")
decoder_cols = joblib.load(PROJECT_ROOT / "data/processed/features/decoder_features.plk")
meteo_features = joblib.load(PROJECT_ROOT / "data/processed/features/meteo_features.plk")
past_features_length = len(encoder_cols) + 1
future_features_length = len(decoder_cols)

# Paramètres
lstm_params = {
    "hidden_size" : 64, 
    "seq_length" : 48,
    "batch_size" : 20,
    "num_layers" : 2, 
    "output_len" : 24,
    "dropout" : 0.25,
    "stride" : 1,
    "num_epochs" : 30,
}

In [40]:
# Pipeline
init_mlflow(config=config, experiment_name="LSTM_Seq2Seq_v1.0-founder") 

run_lstm_pipeline(
    X_train=X_train, 
    y_train=y_train, 
    meteo_features=meteo_features,
    encoder_features=encoder_cols,
    decoder_features=decoder_cols,
    lstm_params=lstm_params)

Epoch [1/30] | Train RMSE: 0.094406 | Val RMSE: 0.092824
Epoch [2/30] | Train RMSE: 0.057230 | Val RMSE: 0.094034
Epoch [3/30] | Train RMSE: 0.051490 | Val RMSE: 0.093557
Epoch [4/30] | Train RMSE: 0.048123 | Val RMSE: 0.092121
Epoch [5/30] | Train RMSE: 0.046340 | Val RMSE: 0.089296
Epoch [6/30] | Train RMSE: 0.044625 | Val RMSE: 0.088589
Epoch [7/30] | Train RMSE: 0.043554 | Val RMSE: 0.086384
Epoch [8/30] | Train RMSE: 0.042341 | Val RMSE: 0.086055
Epoch [9/30] | Train RMSE: 0.041765 | Val RMSE: 0.084009
Epoch [10/30] | Train RMSE: 0.041096 | Val RMSE: 0.082943
Epoch [11/30] | Train RMSE: 0.040605 | Val RMSE: 0.082848
Epoch [12/30] | Train RMSE: 0.040056 | Val RMSE: 0.083265
Epoch [13/30] | Train RMSE: 0.039528 | Val RMSE: 0.080909
Epoch [14/30] | Train RMSE: 0.039285 | Val RMSE: 0.080365
Epoch [15/30] | Train RMSE: 0.038912 | Val RMSE: 0.081146
Epoch [16/30] | Train RMSE: 0.038890 | Val RMSE: 0.079199
Epoch [17/30] | Train RMSE: 0.038394 | Val RMSE: 0.078791
Epoch [18/30] | Train R

🏃 View run LSTM_Seq2Seq_J+1 at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/9/runs/67ba1dd36adb4641a13e4ba70d766cee
🧪 View experiment at: https://mlflow-tracking-server-production.up.railway.app/#/experiments/9


La validation couvre juillet-décembre + quelques jours de janvier, donc elle inclut la fin de l'été (juillet-août, production solaire forte et régulière). Les mois d'hiver ont mécaniquement un RMSE bas, quand solar_mw est proche de 0 la plupart du temps, même un modèle médiocre fait un petit RMSE. Le RMSE val est donc sûrement optimiste.